# RiskFlow: detecting post-outburst scenes at Shisper

A convolutional classifier that gives each Sentinel-2 scene the probability of showing the lake after an outburst. A scene is labelled GLOF when it was acquired within 30 days after one of the four outbursts dated by Lovell and Muhammad (2025). The input is a 5 x 5 km window centred on the lake, resampled to 128 x 128 pixels with six bands.

Two tests, each repeated with five seeds whose probabilities are averaged:
- **main split**: train on the scenes up to 2021 (2021 chooses the number of epochs) and test once on the scenes from 2022 onward;
- **leave one year out**: each year is held out in turn and the number of epochs is chosen on the latest outburst year outside it, so every scene is predicted by a model that never saw its year.

Normalisation is fitted on the training scenes and augmentation is applied to them only, after the split. A water-index rule, which needs no training, is evaluated on the same scenes.

**Input:** `Shisper_lake_S2` (notebook 01). **Output** (`RiskFlow_Shisper`): `riskflow_predictions.csv`, `riskflow_metrics.csv`, `riskflow_seeds.csv`, the main-split models and the cached scenes `scenes_128.npz` (used by notebook 12). GPU runtime (T4), about an hour.

## Setup

In [ ]:
%pip install -q rasterio

import os, re, json, time, shutil, tempfile, warnings
import numpy as np, pandas as pd, tensorflow as tf
import rasterio
from rasterio.warp import transform as warp_transform
from rasterio.windows import from_bounds
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = '/content/drive/MyDrive'
SCENE_DIR = f'{DRIVE}/Shisper_lake_S2'
OUT = f'{DRIVE}/RiskFlow_Shisper'
os.makedirs(OUT, exist_ok=True)

OUTBURSTS = ['2019-06-22', '2020-05-29', '2021-05-15', '2022-05-07']   # 2021: middle of the 10 to 20 May drainage
WINDOW_DAYS = 30                    # GLOF scene: acquired within 30 days after an outburst
IMG, CROP_KM = 128, 5.0
TRAIN_LAST_YEAR, VAL_YEAR = 2021, 2021
AUG_POS, AUG_NEG = 40, 4            # augmented copies of each GLOF and other training scene
MAX_EPOCHS = 60
SEEDS = [42, 1, 2, 3, 4]
THRESHOLDS = (0.5, 0.2)             # 0.2: lower threshold fixed in advance

LAKE = (36.36061, 74.56276)
if os.path.exists(f'{SCENE_DIR}/lake_location.json'):
    j = json.load(open(f'{SCENE_DIR}/lake_location.json')); LAKE = (j['lat'], j['lon'])
print('GPU:', tf.config.list_physical_devices('GPU'), '| lake:', LAKE)

## Scenes and labels

In [ ]:
pat = re.compile(r'^(\d{4}-\d{2}-\d{2})\.tiff?$', re.I)
found = {m.group(1): os.path.join(SCENE_DIR, f) for f in sorted(os.listdir(SCENE_DIR)) if (m := pat.match(f))}
ob = [pd.Timestamp(o) for o in OUTBURSTS]
rows = []
for d, path in sorted(found.items()):
    t = pd.Timestamp(d)
    since = [(t - o).days for o in ob if t >= o]
    until = [(o - t).days for o in ob if o > t]
    s, u = (min(since) if since else np.nan), (min(until) if until else np.nan)
    rows.append(dict(date=d, path=path, year=t.year, label=int(s <= WINDOW_DAYS),
                     days_since_outburst=s, days_to_next_outburst=u))
L = pd.DataFrame(rows)
L['split'] = np.where(L.year <= TRAIN_LAST_YEAR, 'train', 'test')
L['fold'] = L.year.astype(str).where(L.year > 2019, '2018-19')     # the single December 2018 scene joins 2019
print(f'{len(L)} scenes, {L.date.iloc[0]} to {L.date.iloc[-1]} | GLOF scenes:', ', '.join(L[L.label == 1].date))
print(L.groupby('split').label.agg(scenes='count', glof='sum').to_string())

## Read the lake window of every scene

In [ ]:
LOCAL = os.path.join(tempfile.gettempdir(), 'riskflow_scene.tif')

def lake_window(src):
    xs, ys = warp_transform('EPSG:4326', src.crs, [LAKE[1]], [LAKE[0]])
    h = CROP_KM * 500.0
    return from_bounds(xs[0] - h, ys[0] - h, xs[0] + h, ys[0] + h, transform=src.transform).round_offsets().round_lengths()

def read_scene(path, tries=3):
    for k in range(tries):
        try:
            shutil.copyfile(path, LOCAL)                  # Drive sometimes returns short reads
            if os.path.getsize(LOCAL) != os.path.getsize(path):
                raise IOError('incomplete copy')
            with rasterio.open(LOCAL) as src:
                a = src.read(window=lake_window(src), boundless=True, fill_value=0).astype('float32')
                if src.nodata is not None:
                    a[a == src.nodata] = np.nan
            return np.transpose(a, (1, 2, 0))             # (H, W, bands): B4, B3, B2, B8, B11, B12
        except Exception:
            if k == tries - 1:
                raise
            time.sleep(10 * (k + 1))

cache = f'{OUT}/scenes_{IMG}.npz'
C = np.load(cache, allow_pickle=True) if os.path.exists(cache) else None
if C is not None and list(C['dates']) == list(L.date):
    X = C['X']
else:
    X = np.stack([tf.image.resize(np.nan_to_num(read_scene(p)), [IMG, IMG], antialias=True).numpy()
                  for p in L.path]).astype('float32')
    np.savez_compressed(cache, X=X, dates=np.array(L.date.tolist()))
y = L.label.values.astype('float32')
print('scenes:', X.shape)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, d in zip(ax, ['2019-05-31', '2022-05-10', '2023-06-04']):
    if d in set(L.date):
        x = X[L.date.tolist().index(d)][..., :3]
        lo, hi = np.percentile(x, 2), np.percentile(x, 98)
        a.imshow(np.clip((x - lo) / (hi - lo + 1e-6), 0, 1))
    a.set_title(d); a.axis('off')
plt.tight_layout(); plt.show()

## Water-index rule

Open water within 1.5 km of the lake: NDWI above 0.15, B8 below 0.15 and B11 below 0.10 (reflectance). A scene is flagged when a scene of the preceding 60 days had at least 0.05 km² of water and the current one has at most 0.02 km².

In [ ]:
G_, N_, S_ = X[..., 1], X[..., 3], X[..., 4]                   # B3, B8, B11 (reflectance x 10000)
WATER = ((G_ - N_) / (G_ + N_ + 1e-6) > 0.15) & (N_ < 1500) & (S_ < 1000) & (G_ > 0)
yy, xx = np.mgrid[0:IMG, 0:IMG]
PX_KM = CROP_KM / IMG
NEAR = np.hypot(yy - (IMG - 1) / 2, xx - (IMG - 1) / 2) * PX_KM <= 1.5
L['water_km2'] = (WATER & NEAR).sum((1, 2)) * PX_KM ** 2
del G_, N_, S_, WATER

dates = pd.to_datetime(L.date)
L['water_prev_max'] = [L.water_km2[(dates < t) & (dates >= t - pd.Timedelta(days=60))].max() for t in dates]
L['water_rule'] = ((L.water_prev_max >= 0.05) & (L.water_km2 <= 0.02)).astype(int)

## Model and training protocol

In [ ]:
def fit_norm(Xtr):
    return np.percentile(Xtr, 2, axis=(0, 1, 2)), np.percentile(Xtr, 98, axis=(0, 1, 2))

def apply_norm(X, lo, hi):
    return np.clip((X - lo) / (hi - lo + 1e-6), 0, 1).astype('float32')

def augment(X, y, rng):
    xs, ys = [X], [y]
    for xi, yi in zip(X, y):
        for _ in range(AUG_POS if yi == 1 else AUG_NEG):
            a = xi
            if rng.random() < .5: a = a[:, ::-1]
            if rng.random() < .5: a = a[::-1, :]
            a = np.ascontiguousarray(np.rot90(a, int(rng.integers(4))))
            s = int(IMG * rng.uniform(.85, 1.0))                 # random crop, resized back
            r0, c0 = rng.integers(0, IMG - s + 1, 2)
            a = tf.image.resize(a[r0:r0 + s, c0:c0 + s], [IMG, IMG]).numpy()
            a = np.clip(a * rng.uniform(.9, 1.1), 0, 1)          # brightness
            xs.append(a[None]); ys.append(np.array([yi]))
    return np.concatenate(xs).astype('float32'), np.concatenate(ys).astype('float32')

def build():
    m = models.Sequential([
        layers.Input((IMG, IMG, 6)),
        layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(2),
        layers.Flatten(), layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')])
    m.compile(tf.keras.optimizers.Adam(1e-3), 'binary_crossentropy', metrics=['accuracy'])
    return m

def run_protocol(train_mask, val_mask, test_mask, seed, save=None):
    # stage 1: choose the number of epochs on the validation year; stage 2: retrain on all training years
    inner = train_mask & ~val_mask
    tf.keras.utils.set_random_seed(seed); rng = np.random.default_rng(seed)
    lo, hi = fit_norm(X[inner])
    Xa, ya = augment(apply_norm(X[inner], lo, hi), y[inner], rng)
    m = build()
    es = tf.keras.callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True)
    h = m.fit(Xa, ya, validation_data=(apply_norm(X[val_mask], lo, hi), y[val_mask]),
              epochs=MAX_EPOCHS, batch_size=16, callbacks=[es], verbose=0)
    ep = int(np.argmin(h.history['val_loss'])) + 1
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed); rng = np.random.default_rng(seed)
    lo, hi = fit_norm(X[train_mask])
    Xa, ya = augment(apply_norm(X[train_mask], lo, hi), y[train_mask], rng)
    m = build()
    m.fit(Xa, ya, epochs=ep, batch_size=16, verbose=0)
    p = m.predict(apply_norm(X[test_mask], lo, hi), verbose=0).ravel()
    if save:
        m.save(save)
    tf.keras.backend.clear_session()
    return p, ep

## Main split: train to 2021, test 2022 onward

In [ ]:
trB, vaB, teB = (L.year <= TRAIN_LAST_YEAR).values, (L.year == VAL_YEAR).values, (L.year > TRAIN_LAST_YEAR).values
P_MAIN = {}
for s in SEEDS:
    P_MAIN[s], ep = run_protocol(trB, vaB, teB, s, save=f'{OUT}/riskflow_main_seed{s}.keras')
    print(f'seed {s}: {ep} epochs', flush=True)

## Leave one year out

In [ ]:
GLOF_YEARS = sorted(L[L.label == 1].year.unique())
def val_year_for(fold):
    test_years = set(L[L.fold == fold].year)
    return max(yr for yr in GLOF_YEARS if yr not in test_years)

P_LOYO = {}
for s in SEEDS:
    P_LOYO[s] = np.full(len(L), np.nan)
    for f in sorted(L.fold.unique()):
        te = (L.fold == f).values
        P_LOYO[s][te], ep = run_protocol(~te, (L.year == val_year_for(f)).values, te, s)
    print(f'seed {s} done', flush=True)

## Results

In [ ]:
def wilson(k, n, z=1.96):
    # 95 % Wilson interval for k of n
    if n == 0:
        return ''
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; w = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return f'{max(0.0, c - w):.2f} to {min(1.0, c + w):.2f}'

def summary(lab, prob, t=0.5, binary=False):
    lab = np.asarray(lab).astype(int)
    pred = np.asarray(prob) >= 0.5 if binary else np.asarray(prob) > t
    tn, fp, fn, tp = (int(v) for v in confusion_matrix(lab, pred.astype(int), labels=[0, 1]).ravel())
    return dict(TP=tp, FN=fn, FP=fp, TN=tn, caught=f'{tp} of {tp + fn}', caught_ci95=wilson(tp, tp + fn),
                false_alarms=f'{fp} of {fp + tn}', false_alarms_ci95=wilson(fp, fp + tn),
                auc=None if binary else round(float(roc_auc_score(lab, prob)), 3))

PM = np.vstack([P_MAIN[s] for s in SEEDS])
PL = np.vstack([P_LOYO[s] for s in SEEDS])
L['p_main'] = np.nan
L.loc[teB, 'p_main'] = PM.mean(0)
L['p_loyo'], L['p_loyo_min'], L['p_loyo_max'] = PL.mean(0), PL.min(0), PL.max(0)
L['seeds_flagging'] = (PL > 0.5).sum(0)
for s in SEEDS:
    L[f'p_main_seed{s}'] = np.nan
    L.loc[teB, f'p_main_seed{s}'] = P_MAIN[s]
    L[f'p_loyo_seed{s}'] = P_LOYO[s]

rows = []
for test, mask, col in [('main split', teB, 'p_main'), ('leave one year out', np.ones(len(L), bool), 'p_loyo')]:
    lab = L.label.values[mask]
    for t in THRESHOLDS:
        rows.append(dict(test=test, detector=f'RiskFlow, {len(SEEDS)}-seed mean, threshold {t}', **summary(lab, L[col].values[mask], t)))
    rows.append(dict(test=test, detector='water-index rule', **summary(lab, L.water_rule.values[mask], binary=True)))
METRICS = pd.DataFrame(rows)
SEED_TAB = pd.DataFrame([dict(test=test, seed=s, **summary(L.label.values[mask], p))
                         for s in SEEDS for test, mask, p in [('main split', teB, P_MAIN[s]),
                                                              ('leave one year out', np.ones(len(L), bool), P_LOYO[s])]])
L.drop(columns='path').round(6).to_csv(f'{OUT}/riskflow_predictions.csv', index=False)
METRICS.to_csv(f'{OUT}/riskflow_metrics.csv', index=False)
SEED_TAB.to_csv(f'{OUT}/riskflow_seeds.csv', index=False)

cols = ['test', 'detector', 'caught', 'caught_ci95', 'false_alarms', 'false_alarms_ci95', 'auc']
print(METRICS[cols].to_string(index=False))
print('\nsingle seeds (threshold 0.5):')
print(SEED_TAB.groupby('test').agg(caught_min=('TP', 'min'), caught_max=('TP', 'max'), false_alarms_min=('FP', 'min'),
                                   false_alarms_max=('FP', 'max'), auc_min=('auc', 'min'), auc_max=('auc', 'max')).to_string())
print('\nGLOF scenes:')
print(L[L.label == 1][['date', 'days_since_outburst', 'water_km2', 'water_rule', 'p_main', 'p_loyo', 'seeds_flagging']]
      .round(3).to_string(index=False))

In [ ]:
g = L.label == 1
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.scatter(dates[~g], L.p_loyo[~g], s=12, c='#3a6ea5', label='other scene')
ax.errorbar(dates[g], L.p_loyo[g], yerr=[L.p_loyo[g] - L.p_loyo_min[g], L.p_loyo_max[g] - L.p_loyo[g]],
            fmt='^', ms=7, c='#c0392b', capsize=2, label='GLOF scene (bar: range over seeds)')
for o in ob:
    ax.axvline(o, color='#c0392b', lw=.8, ls='--')
ax.axhline(0.5, ls=':', c='0.4')
ax.set_ylabel('P(GLOF), year held out'); ax.set_ylim(-.03, 1.03); ax.legend(fontsize=8, loc='upper right')
plt.tight_layout(); plt.savefig(f'{OUT}/fig_riskflow_loyo.png', dpi=200); plt.show()